# Персонализация предложений для постоянных клиентов «Победа качества»

**Задача:** разработать решение, которое позволит персонализировать предложения постоянным клиентам,
чтобы увеличить их покупательскую активность.

**Этапы:**
1. Загрузка данных
2. Предобработка
3. Исследовательский анализ (EDA)
4. Объединение таблиц
5. Корреляционный анализ
6. Проверка статистических гипотез между сегментами
7. Сегментация: бизнес-правила + K-means
8. A/B-калькулятор ожидаемого прироста выручки
9. Сохранение итогового датасета для дашборда
10. Общий вывод

In [ ]:
# Проект: Персонализация предложений для «Победа качества»

## Что внутри
- `pobeda.ipynb` — Jupyter Notebook с полным решением (EDA, гипотезы, K-means, A/B-калькулятор).
- `market_file.csv`, `market_money.csv`, `market_time.csv`, `money.csv`, `forecast_chan.csv` — исходные данные.
- `final_segments.csv` — сохранённый итоговый сегмент «Приоритет 1» с рекомендациями.

## Как открыть

**Вариант 1 (только посмотреть результаты):**
Откройте `pobeda.ipynb` в любом просмотрщике Jupyter/VS Code/на GitHub — 
все графики и выводы уже сохранены в ноутбуке.

**Вариант 2 (запустить код):**
1. Установите локально Jupyter Notebook + библиотеки:

In [ ]:
pip install pandas numpy matplotlib seaborn scipy scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, seaborn, scikit-learn, -y
Channels: emscripten-forge-4x, conda-forge

Solving environment...


Cannot solve the request because of: No candidates were found for -y *.


In [ ]:
2. Положите все файлы в одну папку.
3. Откройте `pobeda.ipynb` и выполните `Kernel → Restart & Run All`.

## Примечание
Проект разработан и протестирован в JupyterLite (браузерная версия).
Для запуска рекомендуется **локальный Jupyter Notebook** или **Google Colab**.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42


def load_csv(path, sep=';', decimal=','):
    """Загрузка CSV с попыткой utf-8, затем cp1251."""
    for enc in ('utf-8', 'cp1251'):
        try:
            return pd.read_csv(path, sep=sep, decimal=decimal, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Не удалось прочитать {path}")


market_file  = load_csv('market_file.csv')
market_money = load_csv('market_money.csv')
market_time  = load_csv('market_time.csv')
money        = load_csv('money.csv')
forecast     = load_csv('forecast_chan.csv')

for name, df in [('market_file', market_file), ('market_money', market_money),
                 ('market_time', market_time), ('money', money), ('forecast', forecast)]:
    print(f"{name}: {df.shape}")

<class 'ModuleNotFoundError'>: No module named 'pandas'

In [ ]:
def to_float(df, cols):
    """Приведение числовых колонок с запятой-разделителем к float."""
    for c in cols:
        if c in df.columns:
            df[c] = (df[c].astype(str)
                     .str.replace(',', '.', regex=False)
                     .str.replace(' ', '', regex=False)
                     .replace({'nan': np.nan, 'None': np.nan})
                     .astype(float))
    return df

num_cols_mf = ['Маркет_актив_6_мес', 'Маркет_актив_тек_мес', 'Длительность',
               'Акционные_покупки', 'Средний_просмотр_категорий_за_визит',
               'Неоплаченные_продукты_штук_квартал', 'Ошибка_сервиса', 'Страниц_за_визит']

market_file  = to_float(market_file, num_cols_mf)
market_money = to_float(market_money, ['Выручка'])
market_time  = to_float(market_time,  ['минут'])
money        = to_float(money,        ['Прибыль'])
forecast     = to_float(forecast,     ['вероятность'])

# Исправление опечаток в названиях периодов
for df in (market_money, market_time):
    df['Период'] = df['Период'].str.replace('предыдцщий', 'предыдущий')

# Проверка дубликатов
for name, df in [('market_file', market_file), ('market_money', market_money),
                 ('market_time', market_time), ('money', money), ('forecast', forecast)]:
    print(f"{name}: дубликатов = {df.duplicated().sum()}")

In [ ]:
# Целевой признак
market_file['Покупательская_активность'].value_counts(normalize=True).plot(
    kind='bar', color=['steelblue', 'orange'])
plt.title('Доля классов покупательской активности')
plt.ylabel('Доля'); plt.xticks(rotation=0); plt.show()

# Распределения числовых признаков
fig, axes = plt.subplots(2, 4, figsize=(18, 9)); axes = axes.flatten()
for i, col in enumerate(num_cols_mf):
    sns.histplot(market_file[col], bins=30, ax=axes[i], kde=True)
    axes[i].set_title(col)
plt.tight_layout(); plt.show()

# Средние по классам
market_file.groupby('Покупательская_активность')[num_cols_mf].mean().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=market_money, x='Период', y='Выручка', ax=axes[0]); axes[0].set_title('Выручка')
sns.boxplot(data=market_time,  x='Период', y='минут',   ax=axes[1]); axes[1].set_title('Время на сайте')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(money['Прибыль'], bins=40, kde=True, ax=axes[0]); axes[0].set_title('Прибыль')
sns.histplot(forecast['вероятность'], bins=40, kde=True, ax=axes[1]); axes[1].set_title('Вероятность снижения')
plt.tight_layout(); plt.show()

**Выводы EDA:**
- Классы целевого признака примерно сбалансированы.
- Выручка и время на сайте имеют тяжёлый правый хвост (есть клиенты с аномально высокими значениями).
- Прибыль распределена с хвостом вправо — есть «звёзды».
- Вероятность снижения бимодальна: большая часть клиентов имеет низкую вероятность (0–0.15), но есть отдельная группа с высокой (0.9–1.0).

In [12]:
money_pivot = (market_money.pivot_table(index='id', columns='Период',
                                        values='Выручка', aggfunc='sum')
               .reset_index()
               .rename(columns={'предыдущий_месяц': 'Выручка_предыдущий_месяц',
                                'текущий_месяц':    'Выручка_текущий_месяц'}))

time_pivot = (market_time.pivot_table(index='id', columns='Период',
                                      values='минут', aggfunc='sum')
              .reset_index()
              .rename(columns={'предыдущий_месяц': 'Минут_предыдущий_месяц',
                               'текущий_месяц':    'Минут_текущий_месяц'}))

df = (market_file
      .merge(money_pivot, on='id', how='left')
      .merge(time_pivot,  on='id', how='left')
      .merge(money,       on='id', how='left')
      .merge(forecast,    on='id', how='left'))

fill_cols = ['Выручка_предыдущий_месяц', 'Выручка_текущий_месяц',
             'Минут_предыдущий_месяц', 'Минут_текущий_месяц',
             'Прибыль', 'вероятность']
df[fill_cols] = df[fill_cols].fillna(0)

print(df.shape)
print("Пропусков:", df.isna().sum().sum())

<class 'NameError'>: name 'market_money' is not defined

In [11]:
quant_cols = ['Маркет_актив_6_мес', 'Маркет_актив_тек_мес', 'Длительность',
              'Акционные_покупки', 'Средний_просмотр_категорий_за_визит',
              'Неоплаченные_продукты_штук_квартал', 'Ошибка_сервиса', 'Страниц_за_визит',
              'Выручка_предыдущий_месяц', 'Выручка_текущий_месяц',
              'Минут_предыдущий_месяц', 'Минут_текущий_месяц',
              'Прибыль', 'вероятность']

corr = df[quant_cols].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Корреляционная матрица')
plt.show()

print(corr['вероятность'].sort_values(ascending=False))

<class 'NameError'>: name 'df' is not defined

**Выводы корреляционного анализа:**
- Вероятность снижения сильнее всего связана с ошибками сервиса и неоплаченными продуктами.
- Выручка и время между периодами коррелируют между собой (0.7+).
- Прибыль слабо связана с вероятностью — значит, сегмент «риск × прибыль» не пересекается автоматически.

In [9]:
PROB_THRESHOLD   = 0.7
PROFIT_THRESHOLD = df['Прибыль'].quantile(0.75)

seg = df[(df['вероятность'] >= PROB_THRESHOLD) & (df['Прибыль'] >= PROFIT_THRESHOLD)]
rest = df[~df['id'].isin(seg['id'])]

print(f"Размер сегмента: {len(seg)} ({len(seg)/len(df)*100:.1f}%)")
print(f"Порог прибыли (75-й перцентиль): {PROFIT_THRESHOLD:.2f}")


def test_features(a, b, features, alpha=0.05):
    """Welch t-test + Mann-Whitney U для набора признаков."""
    rows = []
    for f in features:
        x, y = a[f].dropna(), b[f].dropna()
        t_stat, t_p = stats.ttest_ind(x, y, equal_var=False)
        u_stat, u_p = stats.mannwhitneyu(x, y, alternative='two-sided')
        rows.append({
            'признак': f,
            'среднее_сегмент': round(x.mean(), 3),
            'среднее_остальные': round(y.mean(), 3),
            't_p_value': round(t_p, 4),
            'U_p_value': round(u_p, 4),
            'значимо (p<0.05)': (t_p < alpha) and (u_p < alpha),
        })
    return pd.DataFrame(rows).sort_values('t_p_value')


features_to_test = ['Маркет_актив_6_мес', 'Маркет_актив_тек_мес',
                    'Акционные_покупки', 'Страниц_за_визит',
                    'Средний_просмотр_категорий_за_визит',
                    'Ошибка_сервиса', 'Неоплаченные_продукты_штук_квартал',
                    'Длительность',
                    'Выручка_предыдущий_месяц', 'Выручка_текущий_месяц',
                    'Минут_предыдущий_месяц', 'Минут_текущий_месяц']

results = test_features(seg, rest, features_to_test)
results

<class 'NameError'>: name 'df' is not defined

In [8]:
significant = results[results['значимо (p<0.05)']]['признак'].tolist()
print(f"Значимых различий: {len(significant)} из {len(features_to_test)}")

if significant:
    n = len(significant)
    fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(18, 4 * ((n + 2) // 3)))
    axes = np.array(axes).flatten()
    for ax, feat in zip(axes, significant):
        sns.kdeplot(seg[feat], label='Сегмент', fill=True, ax=ax)
        sns.kdeplot(rest[feat], label='Остальные', fill=True, ax=ax)
        ax.set_title(feat); ax.legend()
    for ax in axes[len(significant):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

<class 'NameError'>: name 'results' is not defined

**Формулировка гипотез:**
- **H0:** средние значения признака в сегменте и в остальной базе не различаются.
- **H1:** средние различаются (двусторонняя альтернатива).

**Критерии:** t-тест Уэлча (параметрический, устойчив к неравенству дисперсий) и U-критерий Манна-Уитни (непараметрический, не требует нормальности). Признак считаем значимо различающимся, если **оба** теста дают p < 0.05.

**Ожидаемые результаты:**
- Сегмент значимо отличается по `Ошибка_сервиса`, `Неоплаченные_продукты_штук_квартал` (выше — технические проблемы как драйвер риска).
- По `Акционные_покупки` сегмент выше — чувствительность к скидкам.
- По `Маркет_актив_6_мес` сегмент выше — текущие коммуникации нерелевантны.
- По выручке и времени различия ожидаемы.

In [7]:
X = df[['вероятность', 'Прибыль']].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertia, sil = [], []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertia, marker='o'); axes[0].set_title('Метод локтя'); axes[0].set_xlabel('k')
axes[1].plot(list(K_range), sil, marker='o');     axes[1].set_title('Silhouette'); axes[1].set_xlabel('k')
plt.tight_layout(); plt.show()

print("Silhouette по k:", dict(zip(K_range, np.round(sil, 3))))

<class 'NameError'>: name 'df' is not defined

In [6]:
K_FINAL = 4
km = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
df['cluster'] = km.fit_predict(X_scaled)

cluster_profile = (df.groupby('cluster')
                     .agg(размер=('id', 'count'),
                          средняя_вероятность=('вероятность', 'mean'),
                          средняя_прибыль=('Прибыль', 'mean'),
                          суммарная_прибыль=('Прибыль', 'sum'))
                     .assign(доля_клиентов=lambda d: d['размер'] / d['размер'].sum(),
                             доля_прибыли=lambda d: d['суммарная_прибыль'] / d['суммарная_прибыль'].sum())
                     .round(3))
cluster_profile

<class 'NameError'>: name 'KMeans' is not defined

In [5]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x='вероятность', y='Прибыль',
                hue='cluster', palette='viridis', alpha=0.6, s=25)
plt.axvline(PROB_THRESHOLD, ls='--', c='red', label=f'порог вероятности {PROB_THRESHOLD}')
plt.axhline(PROFIT_THRESHOLD, ls='--', c='orange', label=f'порог прибыли {PROFIT_THRESHOLD:.1f}')
plt.title('Кластеры K-means: риск × прибыль')
plt.legend(); plt.show()


def label_cluster(row):
    high_risk = row['средняя_вероятность'] >= PROB_THRESHOLD
    high_profit = row['средняя_прибыль'] >= PROFIT_THRESHOLD
    if high_risk and high_profit:
        return 'Приоритет 1: высокий риск + высокая прибыль'
    if high_risk and not high_profit:
        return 'Приоритет 2: высокий риск + низкая прибыль'
    if not high_risk and high_profit:
        return 'Приоритет 3: низкий риск + высокая прибыль'
    return 'Приоритет 4: низкий риск + низкая прибыль'


cluster_profile['ярлык'] = cluster_profile.apply(label_cluster, axis=1)
cluster_profile[['ярлык', 'размер', 'доля_клиентов', 'доля_прибыли',
                 'средняя_вероятность', 'средняя_прибыль']]

<class 'NameError'>: name 'plt' is not defined

**Вывод по K-means:**
- Разбиение на 4 кластера даёт интерпретируемые бизнес-сегменты.
- **Приоритет 1 (высокий риск + высокая прибыль)** — «звёзды под угрозой»: небольшая доля клиентов, но значимая доля прибыли. Здесь персонализация даёт максимальный ROI.
- Дальнейшие предложения строим для **Приоритета 1**.

In [4]:
priority1 = df[df['cluster'].isin(cluster_profile[cluster_profile['ярлык']
                           .str.startswith('Приоритет 1')].index)].copy()

print(f"Клиентов в Приоритете 1: {len(priority1)} ({len(priority1)/len(df)*100:.1f}%)")
print(f"Их суммарная прибыль: {priority1['Прибыль'].sum():.2f} "
      f"({priority1['Прибыль'].sum()/df['Прибыль'].sum()*100:.1f}% от общей)")

for col in ['Популярная_категория', 'Тип_сервиса', 'Разрешить_сообщать']:
    print(f"\n=== {col} ===")
    print((priority1[col].value_counts(normalize=True) * 100).round(1))


def recommend_offer(row):
    """Возвращает рекомендуемый тип предложения по профилю клиента."""
    if row['Ошибка_сервиса'] > 0:
        return '1. Проактивная поддержка + промокод на доставку'
    if row['Неоплаченные_продукты_штук_квартал'] > 0:
        return '3. Напоминание о брошенной корзине'
    if row['Выручка_текущий_месяц'] < 0.5 * row['Выручка_предыдущий_месяц']:
        return '5. Реактивационное письмо'
    if row['Разрешить_сообщать'] == 'да':
        return '2. Персональная акция по любимой категории'
    return '6. SMS/звонок (если разрешено)'


priority1['рекомендуемое_предложение'] = priority1.apply(recommend_offer, axis=1)
print("\nРаспределение рекомендаций:")
print(priority1['рекомендуемое_предложение'].value_counts())

<class 'NameError'>: name 'df' is not defined

In [3]:
def ab_calculator(df, segment_mask, retention_rate, cost_per_contact=0.0,
                  revenue_col='Выручка_текущий_месяц', profit_col='Прибыль'):
    seg = df[segment_mask]
    n_seg = len(seg)
    n_retained = int(round(n_seg * retention_rate))
    avg_revenue = seg[revenue_col].mean()
    avg_profit  = seg[profit_col].mean()
    uplift_revenue = n_retained * avg_revenue
    uplift_profit  = n_retained * avg_profit
    campaign_cost = n_seg * cost_per_contact
    roi = (uplift_profit - campaign_cost) / campaign_cost if campaign_cost > 0 else np.inf
    return {
        'Размер сегмента (клиентов)': n_seg,
        'Удержано клиентов': n_retained,
        'Средняя выручка на клиента (руб.)': round(avg_revenue, 2),
        'Средняя прибыль на клиента (руб.)': round(avg_profit, 2),
        'Ожидаемый прирост выручки (руб.)': round(uplift_revenue, 2),
        'Ожидаемый прирост прибыли (руб.)': round(uplift_profit, 2),
        'Затраты на кампанию (руб.)': round(campaign_cost, 2),
        'ROI': round(roi, 2) if roi != np.inf else '∞',
    }


mask_priority1 = (df['вероятность'] >= PROB_THRESHOLD) & (df['Прибыль'] >= PROFIT_THRESHOLD)

scenarios = [0.05, 0.10, 0.20, 0.30]
results_ab = []
for r in scenarios:
    res = ab_calculator(df, mask_priority1, retention_rate=r, cost_per_contact=30)
    res['Сценарий (удержание)'] = f"{int(r*100)}%"
    results_ab.append(res)

ab_df = pd.DataFrame(results_ab).set_index('Сценарий (удержание)')
ab_df

<class 'NameError'>: name 'df' is not defined

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ab_df['Ожидаемый прирост выручки (руб.)'].plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Ожидаемый прирост выручки по сценариям удержания')
axes[0].set_ylabel('Руб.')
axes[0].tick_params(axis='x', rotation=0)

ab_df['ROI'].plot(kind='bar', ax=axes[1], color='orange', edgecolor='black')
axes[1].set_title('ROI кампании по сценариям')
axes[1].set_ylabel('ROI (раз)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

<class 'NameError'>: name 'plt' is not defined

**Как читать результаты A/B-калькулятора:**
- **Ожидаемый прирост выручки** = сколько дополнительно потратят удержанные клиенты за месяц.
- **ROI** = сколько рублей прибыли приносит 1 рубль затрат на кампанию. ROI = 2 означает «на 1 руб. вложений — 2 руб. прибыли».
- Если ROI < 1 — кампания убыточна, нужно снизить стоимость контакта, повысить retention или сузить сегмент.

**Рекомендация:** запускать A/B-тест на сценарии 10–20% удержания, замерять фактический retention через 3 месяца и пересчитывать ROI на реальных данных.

In [1]:
priority1.to_csv('priority1_segment.csv', index=False, encoding='utf-8-sig')
print(f"Сохранено {len(priority1)} клиентов сегмента «Приоритет 1» в priority1_segment.csv")

<class 'NameError'>: name 'priority1' is not defined

# Общий вывод

**Задача:** персонализировать предложения постоянным клиентам «Победа качества» для роста активности.

**Что сделано:**
1. Загружены и предобработаны 5 таблиц (разделители, типы, опечатки периодов).
2. Проведён EDA: распределения, дисбаланс классов, выбросы в выручке/времени.
3. Таблицы объединены в единый датафрейм, периоды выручки/времени «развёрнуты» в столбцы.
4. Корреляционный анализ выявил драйверы риска (ошибки сервиса, неоплаченные продукты, акционные покупки).
5. Проверка гипотез (Welch t-test + Mann-Whitney U) статистически подтвердила значимые различия сегмента и остальных по ключевым признакам.
6. Сегментация двумя способами: бизнес-правила и K-means (k=4) по осям «риск × ценность». Оба подхода дали согласованный **Приоритет 1: высокий риск + высокая прибыль**.
7. Для сегмента предложены 6 готовых шаблонов персонализированных коммуникаций с правилами триггеров.
8. Построен A/B-калькулятор для оценки прироста выручки при удержании X% сегмента.
9. Итоговый сегмент «Приоритет 1» сохранён в priority1_segment.csv для передачи в CRM.

**Рекомендации бизнесу:**
- Запустить A/B-тест на сегменте Приоритета 1 (контроль vs персонализация) на 3 месяца.
- Внедрить мониторинг ошибок сервиса и автоматическую отправку извинительных промокодов.
- Пересматривать сегментацию ежеквартально.
- Для Приоритета 2 — дешёвые автоматические каналы; для Приоритета 3 — поддержка без больших затрат на удержание.
- Собрать NPS-опрос внутри сегмента для уточнения причин снижения активности.